# ClaimGuard — session 4: multi-hop verification on HoVer (about 1 h)

The topic asks for a reasoning module over several documents and for multi-hop claims.
This notebook runs `scripts/run_multihop.py`:
- **E1** gold articles given: read one by one (ClaimGuard) vs together (joint reader), by number of hops
- **E2** retrieval: one round vs two rounds (iterative), share of claims with ALL needed articles found
- **E2b** end to end: iterative retrieval + joint reader
- **E3** fine-tune the joint reader on HoVer train claims (never on dev)

Inputs: **claimguard-code** (upload the NEW `claimguard.zip` as a new version) **and** the session-2 output with the
verifier (*Add Input → Your Work → Notebooks →* session 2), or `verifier.zip` as a dataset. GPU T4 x2, Internet on.
Then *Save Version → Save & Run All*.

## 0. Setup

In [ ]:
import glob, json, os, shutil, subprocess, time, urllib.request
from pathlib import Path

candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
              if "claimguard" in Path(p).read_text()]
assert candidates, "claimguard.zip not found: add it as an input dataset"
SRC = candidates[0]
WORK = Path("/kaggle/working/claimguard")
if not WORK.exists():
    shutil.copytree(SRC, WORK)
os.chdir(WORK)
assert (WORK / "scripts/run_multihop.py").exists(), "old claimguard.zip: upload the new version"
print("project copied from", SRC)

In [ ]:
!pip install -q -r requirements.txt && pip install -q -e .
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: set the accelerator to GPU T4 x2")

In [ ]:
# run a shell command, log time and success, never stop the notebook
STATUS_FILE = WORK / "results" / "run_status_multihop.json"
STATUS_FILE.parent.mkdir(exist_ok=True)
status = []

def run(name, cmd):
    print(f"\n===== {name} =====\n$ {cmd}", flush=True)
    start = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=WORK, text=True, capture_output=True)
    minutes = round((time.time() - start) / 60, 1)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-4000:])
    status.append({"step": name, "ok": proc.returncode == 0, "minutes": minutes,
                   "error": None if proc.returncode == 0 else proc.stderr[-1500:]})
    STATUS_FILE.write_text(json.dumps(status, indent=2))
    print(f"-> {'OK' if proc.returncode == 0 else 'FAILED'} in {minutes} min", flush=True)
    return proc.returncode == 0

In [ ]:
# my SciFact verifier (session 2)
import zipfile
zips = glob.glob("/kaggle/input/**/verifier.zip", recursive=True)
dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/verifier/config.json", recursive=True)]
if zips:
    zipfile.ZipFile(zips[0]).extractall(WORK / "models")
elif dirs:
    shutil.copytree(dirs[0], WORK / "models" / "verifier", dirs_exist_ok=True)
assert (WORK / "models/verifier/config.json").exists(), "verifier not found: add the session-2 output or verifier.zip"
print("verifier OK")

## 1. HoVer and its Wikipedia

In [ ]:
HOVER = WORK / "data/raw/hover"
HOVER.mkdir(parents=True, exist_ok=True)

def fetch(url, dest, timeout=60):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        print("ok", dest.name, round(dest.stat().st_size / 1e6, 1), "MB")
        return True
    except Exception as err:
        print("FAILED", url, type(err).__name__, err)
        if dest.exists():
            dest.unlink()
        return False

RAW = "https://raw.githubusercontent.com/hover-nlp/hover/main/data/hover/"
for split in ("train", "dev"):
    fetch(RAW + f"hover_{split}_release_v1.1.json", HOVER / f"hover_{split}_release_v1.1.json")
# TF-IDF candidates of the HoVer authors (used to build the closed pool); optional
fetch("https://nlp.cs.unc.edu/data/hover/dev_tfidf_doc_retrieval_results.json",
      HOVER / "dev_tfidf_doc_retrieval_results.json", timeout=120)

In [ ]:
# the Wikipedia text: first the HoVer authors' SQLite file, else the BEIR copy of the same dump
WIKI = None
db = HOVER / "wiki_wo_links.db"
t0 = time.time()
if fetch("https://nlp.cs.unc.edu/data/hover/wiki_wo_links.db", db, timeout=300):
    WIKI = db
else:
    from huggingface_hub import hf_hub_download
    WIKI = Path(hf_hub_download("BeIR/hotpotqa", "corpus.jsonl.gz", repo_type="dataset", local_dir=str(HOVER)))
print("wiki source:", WIKI, round(WIKI.stat().st_size / 1e9, 2), "GB in", round((time.time() - t0) / 60, 1), "min")

## 2. The experiment (E1, E3, E2, E2b)

In [ ]:
ok = run("multihop", f"python scripts/run_multihop.py --hover-dir {HOVER} --wiki {WIKI} "
         "--verifier models/verifier --train-joint --out results")

## 3. Summary

In [ ]:
r = json.loads((WORK / "results/multihop.json").read_text())
print("dev claims:", r["n_dev"], "| gold coverage:", round(r["gold_coverage"], 3), "| pool:", r.get("pool_kind"), r.get("pool_size"))
for setting in ("e1_gold", "e1_distractors"):
    print("\n", setting, "(accuracy, balanced sample)")
    for model, block in r[setting].items():
        for reading in ("independent_noisy_or", "independent_max", "joint"):
            row = [f"{block[reading][h]['accuracy']['value']:.3f}" for h in ("all", "2_hops", "3_hops", "4_hops")]
            print(f"  {model:20s} {reading:22s} all/2/3/4 hops:", " ".join(row))
for mode in ("single_shot", "iterative"):
    b = r["e2_retrieval"][mode]["all_found@10"]
    print("\nall articles in top 10,", mode, [round(b[h]["value"], 3) for h in ("all", "2_hops", "3_hops", "4_hops")])
print("\nend to end:", {k: round(v["all"]["accuracy"]["value"], 3) for k, v in r["e2b_end_to_end"].items() if isinstance(v, dict)})

In [ ]:
# everything to download in one file (the HoVer reader itself is big, it stays out of the zip)
out = Path("/kaggle/working/results_multihop.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (WORK / "results").rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(WORK))
    log = WORK / "models/hover_joint_reader/train_log.json"
    if log.exists():
        z.write(log, "results/hover_joint_reader_train_log.json")
print(out, round(out.stat().st_size / 1e6, 1), "MB")
print(json.dumps(status, indent=2))